# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## 제출 정보
- 이름:장충만
- GitHub ID:candle1030
- 작성일:09-22
- 최종 Notebook URL:

## 1. 예측 문제 정의
- Target: order_total (같은 order_id의 line_total 합계)
- 예측 단위: 주문 1건
- 예측 시점: 주문이 접수되는 시점 (주문 상세가 확정되기 전)
- 예측 시점에 알 수 있는 정보: 주문 메타데이터(주문월, 요일), 고객의 비식별 특성(나이, 결제수단, 성별, 도시)
- 예측 시점에 알 수 없는 정보: 주문 상세의 수량, 단가, 항목별 금액, 주문 상태

### 나의 해석과 판단
실제 업무에서는 주문이 완료되기 전, 즉 상세 품목·수량·금액이 확정되기 전 시점에 예측이 필요한 경우가 많다(예: 재고 준비, 매출 예측). 만약 주문 상세 데이터를 feature로 쓰면 사실상 정답의 재료를 입력에 포함시키는 것이므로, 예측 시점에 실제로 확보 가능한 정보만 사용하는 것이 이 문제 설정의 핵심이라고 판단했다.

> 🤖 *이 섹션의 "나의 해석과 판단"은 LLM 초안을 받아 검토 후 사용함.*


## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [2]:
from pathlib import Path
import sys
import pandas as pd

def _find_project_root(start: Path) -> Path:
    """Walk upward until a folder containing src/regression.py is found.

    This works no matter which folder the notebook itself lives in
    (notebooks/, chapter09/, or the project root itself).
    """
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "regression.py").exists():
            return candidate
    return start

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = _find_project_root(CURRENT_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)


PROJECT_ROOT: C:\dev\llm-data-analysis-course
PROCESSED_DIR: C:\dev\llm-data-analysis-course\data\processed


## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [3]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


## 2. 검증된 전처리 데이터를 불러옵니다

In [4]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [5]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-07-09,bank_transfer,completed,7,2,361000,M,67,대구
1,79,62,2025-07-11,bank_transfer,refunded,7,4,283000,F,63,광주
2,12,98,2025-07-12,kakao_pay,completed,7,5,444000,F,61,인천
3,169,9,2025-07-12,kakao_pay,completed,7,5,448000,M,69,서울
4,152,62,2025-07-13,naver_pay,refunded,7,6,985000,F,63,광주


Model rows: 300


## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity x unit_price
order_total = 같은 order_id의 line_total 합계
```

- line_total 검증 결과: build_regression_dataset 실행 시 자동 검증 통과 (불일치 없음)
- orders ↔ target 관계 검증 결과: one_to_one 확인 (model_data 300행 = orders 300행, 중복/누락 없음)
- orders → customers 관계 검증 결과: many_to_one 확인 (model_data.head() 출력에서 customer_id 결측 없이 모두 매핑됨)

### 나의 해석과 판단
관계 검증 없이 단순 merge만 하면, 부모 레코드가 없는 주문(customer_id가 customers에 없는 경우 등)이 조용히 NaN으로 채워지거나 해당 행이 누락되어, 실제보다 적은/왜곡된 행으로 모델을 학습하게 될 수 있다. 이러면 성능 지표 자체가 왜곡되고, 실제 존재하는 주문이 예측 대상에서 빠지는 문제로 이어질 수 있다.


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 4. Feature Audit을 확인합니다

In [6]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| order_dayofweek | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| age | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| payment_method | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| gender | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| city | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| avg_unit_price | X | O | X | 미사용 | 주문 상세에서 만든 목표 대리 변수 |
| customer_id | X | X | O | 미사용 | 고객 식별자 |
| item_count | X | X | O | 미사용 | 주문 상세가 확인된 뒤 계산되는 사후 집계 |
| line_total | X | O | X | 미사용 | 목표값을 구성하는 주문 상세 금액 |
| order_id | X | X | O | 미사용 | 주문 식별자 |
| order_status | X | X | O | 미사용 | 예측 시점 이후에 확정될 수 있는 사후 정보 |
| order_total | X | O | X | 미사용 | 예측 대상 자체 |
| product_id | X | X | O | 미사용 | 주문 상세가 확인되어야 알 수 있는 식별자 |
| quantity | X | O | X | 미사용 | 목표값 계산 재료 |
| total_quantity | X | O | X | 미사용 | 주문 상세에서 만든 목표 대리 변수 |
| unit_price | X | O | X | 미사용 | 목표값 계산 재료 |

- 금지 feature overlap 개수: 0개 (FEATURE_COLUMNS에는 forbidden 항목이 없고, Validation의 forbidden_feature_overlap 값도 0으로 확인됨)
- 가장 위험하다고 판단한 feature: avg_unit_price, total_quantity
- 이유: order_total 자체는 이름만 봐도 명백히 제외 대상이라 실수할 위험이 적지만, avg_unit_price·total_quantity는 이름만 보면 그럴듯한 feature처럼 보여서 무심코 넣기 쉽다. 실제로는 order_items를 집계해서 만든 값이라 order_total과 사실상 같은 정보(고객이 얼마나 많이/비싸게 샀는지)를 담고 있는 대리 변수다.

![Leakage 점검](images/step02_leakage.png)


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [7]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-07-09,2026-04-13
1,test,56,18.67,2026-04-14,2026-07-08


Chronological split gate: PASS


In [8]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

Train: (244, 6) Final Test: (56, 6)


## 4. Train / Final Test 분할

- Train 시작일: 2025-07-09
- Train 종료일: 2026-04-13
- Train 행 수: 244
- Final Test 시작일: 2026-04-14
- Final Test 종료일: 2026-07-08
- Final Test 행 수: 56
- 같은 달력 날짜 중복 여부: 없음 (Train 종료일 2026-04-13 < Final Test 시작일 2026-04-14, 코드의 `Chronological split gate: PASS`로도 확인됨)

### 분할 판단
이 문제는 미래 시점의 주문 금액을 예측하는 것이 목적이므로, 무작위 분할을 하면 미래 데이터의 패턴이 우연히 Train에 섞여 실제로는 예측 시점에 알 수 없는 정보가 학습에 들어가는 leakage가 생긴다. 시간 순서로 나눠야 실제 운영 환경(과거 데이터로 학습해서 미래 데이터에 적용)과 동일한 조건에서 성능을 평가할 수 있다.

![시간 분할](images/step04_split.png)


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [9]:
models = make_regression_models(random_state=42)
print(list(models))

['Baseline Mean', 'Linear Regression', 'Random Forest']


## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [10]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,437352.803929,26318.793429,-0.109687
2,Linear Regression,5,461386.286905,77792.799324,-0.310517


## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 423,004.81 | 44,153.04 | -0.0228 | |
| Linear Regression | 461,386.29 | 77,792.80 | -0.3105 | 표준편차가 세 모델 중 가장 큼 (fold별 변동 큼) |
| Random Forest | 437,352.80 | 26,318.79 | -0.1097 | 표준편차가 세 모델 중 가장 작음 (상대적으로 안정적) |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest
- 선택 기준: Linear Regression과 Random Forest 중 cv_MAE_mean이 더 낮은 모델 (437,352.80 < 461,386.29)
- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단
CV MAE 평균만 보면 Random Forest가 Linear Regression보다 근소하게 낫다. 하지만 표준편차를 같이 보면 Linear Regression은 77,792로 Random Forest(26,318)보다 3배 가까이 fold 간 변동이 크다. 즉 Linear Regression은 특정 fold에서 우연히 잘 맞았을 가능성이 있어, 평균만으로 판단하면 운이 좋았던 결과를 실력으로 착각할 수 있다. 다만 두 모델 모두 Baseline Mean(423,004.81)보다 CV MAE가 높다는 점은 그대로 남는 한계다.


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [11]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [12]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47


### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266.19 | 443,449.65 | 522,189.70 | -0.0011 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 326,438.48 | 458,819.92 | 548,190.45 | -0.1033 | -3.47% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
Frozen Model(Random Forest)의 Test MAE(458,819.92)가 Baseline(443,449.65)보다 오히려 높다. 개선율은 -3.47%로, Baseline보다 못하다. Train MAE(326,438)는 Test MAE(458,820)보다 훨씬 낮아 Train-Test 간 격차가 크다. Test R²도 -0.1033으로 음수다.

### 나의 해석과 판단
- Frozen Model이 baseline보다 실제로 개선되지 않았다 (오히려 -3.47% 악화됐다).
- Test RMSE(548,190)가 Test MAE(458,820)보다 꽤 크다는 건, 일부 큰 오차 사례가 전체 평가를 끌어올리고 있다는 뜻으로 해석할 수 있다.
- R²가 음수라는 건 "그냥 평균값으로 예측하는 것보다도 못하다"는 의미로 해석할 수 있다.

### 업무·분석적 의미
현재 모델은 평균 주문 금액을 그대로 찍는 것보다도 못한 수준이라, 실제 업무(재고 준비, 매출 예측 등)에 그대로 쓰면 오히려 의사결정을 오도할 위험이 있다. 지금 허용된 feature(주문월, 요일, 나이, 결제수단, 성별, 도시)만으로는 order_total의 변동을 설명할 신호가 부족하다고 볼 수 있다.


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 10. 실제값·예측값·잔차를 진단합니다

In [13]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-06-15,1877000,605528.715709,1.271471e+06,1.271471e+06,Random Forest
1,21,2026-06-19,1767000,756889.654706,1.010110e+06,1.010110e+06,Random Forest
2,145,2026-05-27,1863000,861784.708271,1.001215e+06,1.001215e+06,Random Forest
3,96,2026-05-11,1690000,694946.179725,9.950538e+05,9.950538e+05,Random Forest
4,183,2026-05-23,1685000,745836.316118,9.391637e+05,9.391637e+05,Random Forest
5,168,2026-06-11,10000,897415.394314,-8.874154e+05,8.874154e+05,Random Forest
6,15,2026-05-23,1677000,849179.542426,8.278205e+05,8.278205e+05,Random Forest
7,179,2026-07-08,86000,904745.854532,-8.187459e+05,8.187459e+05,Random Forest
8,282,2026-04-24,1566000,758941.292777,8.070587e+05,8.070587e+05,Random Forest
9,73,2026-06-14,72000,871636.144070,-7.996361e+05,7.996361e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


## 7. 대표 오차 사례

### 사례 1
- 실제값: 1,877,000원
- 예측값: 605,528.72원
- residual: 1,271,471.28
- absolute error: 1,271,471.28
- 관찰: 실제로는 188만원대의 고액 주문인데, 모델은 60만원 수준으로 크게 과소예측했다.
- 가능한 이유 후보: 현재 feature(주문월, 요일, 나이, 결제수단, 성별, 도시)에는 "이 고객/주문이 고액 구매 패턴인지"를 알려주는 정보가 없어, 고액 주문을 구분할 신호가 부족했을 수 있다.
- 추가 확인: 예측 시점에 합법적으로 알 수 있는 범위에서, 고객별 과거 구매 규모 같은 추가 정보가 있는지 확인이 필요하다.

### 사례 2
- 실제값: 10,000원
- 예측값: 897,415.39원
- residual: -887,415.39
- absolute error: 887,415.39
- 관찰: 실제로는 1만원짜리 매우 작은 주문인데, 모델은 90만원 가까이로 크게 과대예측했다.
- 가능한 이유 후보: Random Forest가 Train 데이터의 평균적인 주문 규모(대부분 수십만원대)에 끌려가서, 소액 주문도 평균 근처로 예측하는 경향이 있을 수 있다.
- 추가 확인: 이런 소액 주문이 Train 데이터에도 충분히 있었는지, 있었다면 왜 학습이 안 됐는지 확인이 필요하다.

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 11. 자동 Validation Evidence를 만듭니다

In [14]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성.

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목
- 없음


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [15]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> C:\dev\llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\dev\llm-data-analysis-course\reports\ch09_regression_split_summary.csv
feature_audit -> C:\dev\llm-data-analysis-course\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\dev\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\dev\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\dev\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
validation -> C:\dev\llm-data-analysis-course\reports\ch09_regression_validation.csv
checklist -> C:\dev\llm-data-analysis-course\reports\ch09_regression_checklist.csv
report -> C:\dev\llm-data-analysis-course\reports\ch09_regression_report.md
actual_vs_predicted_figure -> C:\dev\llm-data-analysis-course\reports\figures\ch09_actual_vs_predicted.png
residual_figure -> C:\dev\llm-data-analysis-course\reports\figures\ch09_resid

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
`predictions_internal`, `model_data_internal`에는 order_id가 그대로 들어있어 특정 고객의 특정 주문을 역추적할 수 있다. 반면 공개 보고서(`ch09_regression_report.md`, figures)는 집계된 지표와 그래프만 담고 있어 개인/주문을 특정할 수 없다. 그래서 order_id가 포함된 파일은 내부 진단용으로만 두고, 외부에 공유하는 자료에서는 제외해야 한다.


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [16]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: 성공 (에러 없이 끝까지 실행됨)
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: PASS (6개 체크 모두 PASS)
- Notebook 결과와 스크립트 결과의 일관성: 노트북 셀 13에서 `run_regression_analysis()`를 새로 호출한 결과도 Selected Model = Random Forest, Validation 전부 PASS로 노트북 앞부분과 동일하게 재현됨을 확인함. (터미널에서 `python scripts/run_regression_analysis.py`도 별도로 한 번 더 실행해서 대조하는 것을 권장.)


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 15. 완료 체크

- [ ] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [ ] Target과 예측 시점을 설명할 수 있다.
- [ ] 금지 feature가 입력에 없음을 확인했다.
- [ ] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [ ] 전처리는 Pipeline 내부에서 학습된다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했다.
- [ ] Final Test 전에 Selected Model을 고정했다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [ ] MAE, RMSE, R²를 함께 읽었다.
- [ ] 내부 식별자 결과와 공개 보고서를 구분했다.
- [ ] Validation Evidence가 모두 PASS다.

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Frozen Model(Random Forest)의 Final Test MAE가 Baseline Mean보다 오히려 3.47% 더 나쁘다 — 평균값으로 찍는 것보다도 못한 수준이다.
2. Test R²가 -0.1033으로 음수다 — 현재 feature들이 order_total의 변동을 설명하는 데 거의 도움이 안 된다는 뜻이다.
3. 누수 검증(Feature Audit)과 Validation Evidence는 모두 통과했으므로, 이 결과는 코드 버그나 데이터 누수가 아니라 "현재 허용된 feature만으로는 신호가 부족하다"는 것을 보여준다.

### 업무적 위험
지금 이 모델을 그대로 운영에 쓰면, 평균값을 쓰는 것보다 예측이 더 부정확해서 재고·매출 계획 등의 의사결정을 오히려 악화시킬 수 있다.

### 현재 데이터의 한계
예측 시점에 허용된 feature(주문월, 요일, 나이, 결제수단, 성별, 도시)가 주문 금액과 직접적인 인과관계가 약한 인구통계/시점 정보 위주라서, 고객의 과거 구매 패턴이나 상품 카테고리 같은 더 강한 신호가 빠져 있다.

### 다음 개선 우선순위
1. 예측 시점에 합법적으로 알 수 있는 추가 feature 발굴 (예: 이번 주문 이전 시점까지의 고객 누적 구매 정보 — leakage 없이 시점을 엄격히 지켜서)
2. 표본 수(300건)가 적어 CV 표준편차가 큰 편이므로, 데이터 수집 기간을 늘려 재검증
3. Random Forest 하이퍼파라미터 튜닝이나 Gradient Boosting 등 다른 모델 계열 추가 비교


> 🤖 *이 섹션의 해석/판단 문장은 LLM 초안을 받아 검토 후 사용함 (실제 실행 결과 근거).*


## 최종 체크

- [ ] Target과 예측 시점을 정의했습니다.
- [ ] Leakage Audit을 수행했습니다.
- [ ] Target 계산과 관계 검증을 확인했습니다.
- [ ] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [ ] 전처리가 Pipeline 내부에서 학습됩니다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [ ] Final Test 전에 Selected Model을 고정했습니다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [ ] MAE, RMSE, R²를 함께 해석했습니다.
- [ ] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [ ] Validation Evidence가 모두 PASS입니다.
- [ ] Internal 결과와 공개 결과를 구분했습니다.
- [ ] 낮은 성능도 숨기지 않았습니다.
- [ ] 전체 스크립트를 재실행했습니다.
- [ ] 최종 Notebook URL을 제출합니다.

> ✍️ **[내 답안 작성]** 아래 항목을 실제 실행 결과를 보고 직접 채워 넣으세요. 이미지 경로(`images/...`)는 이 노트북과 같은 폴더에 `images/` 하위 폴더를 만들고 스크린샷을 넣으면 그대로 동작합니다.
